# Notebook de obtención, limpieza y transformación de datos — Fase 2

**Proyecto:** Factores sociodemograficos y riesgo cardiovascular (ENS 2016-2017)

---

### ¿Qué es un *pipeline* de preprocesamiento?

Un *pipeline* es una secuencia ordenada de pasos por los que pasan los datos crudos hasta
quedar listos para analizar o modelar. En este notebook el flujo es:

**Obtener → Explorar → Limpiar → Transformar → Escalar → Validar**

Cada paso se implementa como una **función** reutilizable. Trabajar con funciones (en vez
de copiar y pegar código) hace el proceso más ordenado, más fácil de corregir y permite
reutilizar la misma lógica sobre cualquier columna sin reescribirla.

## Introducción

Según el Ministerio de Salud de Chile, las enfermedades cardiovasculares representan una de las principales causas de morbimortalidad en el país. Este conjunto de datos de la Encuesta Nacional de Salud (ENS) 2016–2017 permite investigar cómo se asocian factores sociodemográficos como la edad, el sexo, la educación y el nivel socioeconómico con los principales factores de riesgo cardiovascular

**Objetivo general (Fase 2).** Construir un *pipeline* reproducible que deje el *dataset*
limpio, codificado y escalado, listo para el modelado posterior.

**Objetivos específicos.**
- Obtener y explorar los datos verificando su estructura y calidad.
- Limpiar gestionando rigurosamente los valores nulos.
- Transformar las variables categóricas nominales con codificación One-Hot.
- Estandarizar las variables continuas.
- Validar técnicamente el resultado.

**Atributos:** `IdEncuesta`, `FechaInicioF1`, `Estrato`, `Conglomerado`, `Fexp_F1F2p_Corr`,`anos_estudio_MINSAL_1`,
`as27`, `as28`,`HTA`,`di3`,`di2`,`IMC`,`GPAQ`.

**Variable objetivo:** factores de riesgo cardiovascular (hipertensión, diabetes, high colesterol)

## TODO: Revisar la variable objetivo, esta es la que usa el profesor para su ejemplo.
objetivo `stroke` (1 = sufrió accidente cerebrovascular, 0 = no).

### Librerías utilizadas

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Cálculo numérico y manejo de arreglos. |
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `matplotlib` | Generar los gráficos (*boxplots*). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar
dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

In [ ]:
import numpy as np                  # cálculo numérico y operaciones vectorizadas
import pandas as pd                 # estructuras tabulares: Series y DataFrame
import matplotlib.pyplot as plt     # gráficos de control
from sklearn import preprocessing    # LabelEncoder y OneHotEncoder
from sklearn.preprocessing import StandardScaler   # estandarización z-score

# Reproducibilidad: fijamos la semilla aleatoria del entorno
np.random.seed(42)

%matplotlib inline

## Configuración del entorno y de las rutas

Antes de cargar nada conviene comprobar el entorno y dejar declaradas las rutas.
Se usan rutas **relativas** a la raíz del repositorio, coherentes con la
estructura creada en la Fase 1: una ruta absoluta como `C:/Users/...` funciona en
un solo computador del mundo.

In [ ]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("numpy", np), ("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
DIR_CRUDO = Path("data/raw")            # datos originales: nunca se modifican
DIR_PROCESADO = Path("data/processed")  # resultado del pipeline
DIR_DOCS = Path("docs")                 # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "ens2016.xlsx"
print("\nArchivo esperado en:", ARCHIVO)

### Procedencia del conjunto de datos

| Campo | Valor |
| --- | --- |
| Título | Stroke Prediction Dataset ...completar titulo dataset |
| Autor | completar |
| Plataforma | completar |
| Enlace | `https://www......dataset` |
| Estructura esperada | 5.110 filas × 12 columnas · 201 nulos en `bmi` |
| Unidad de observación | Un paciente |

**Referencia en APA 7 para el informe:**

> fedesoriano. (2021). *Stroke Prediction Dataset* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

Documentar la procedencia no es un trámite: sin ella, nadie puede verificar sobre
qué versión del archivo se trabajó.

In [ ]:
# Reemplaza el nombre por el de tu archivo real
# Buscar el dataset desde la carpeta superior del proyecto
base = Path.cwd().parent

archivo = next(
    base.rglob("ens2016.xlsx"),
    None
)

if archivo is None:
    raise FileNotFoundError(
        f"No se encontró el dataset dentro de:\n{base}"
    )

print("Dataset encontrado en:")
print(archivo)

df = pd.read_excel(archivo)

print(f"\nDataset cargado correctamente")
print(f"Dimensiones: {df.shape}")

df.head()

## 1. Obtención de los datos

**Qué hace este paso.** Lee el archivo XLSX y lo carga en un `DataFrame` (la tabla con la
que trabajaremos).

**Por qué con una función.** Encapsulamos la lectura en `cargar_datos(ruta)` con un bloque
`try / except`. Así, si el archivo no existe, en lugar de un error técnico confuso, el
usuario recibe un mensaje claro indicando qué revisar. La función, además, imprime las
dimensiones cargadas, lo que sirve como primera verificación de que el archivo se leyó bien.

Es una función (def) que recibe un parámetro, ruta (el nombre del archivo), y se encarga de leer el XLSX. El texto entre """...""" es el docstring: solo documentación, no se ejecuta. El bloque try/except maneja errores: intenta leer el archivo con pd.read_xls(ruta) y, si no lo encuentra, en vez de romperse muestra un mensaje claro de qué revisar (usando un f-string, que inserta el valor de ruta en el texto). Luego print informa el tamaño con df.shape[0] (filas) y df.shape[1] (columnas) —de ahí el "5110 filas y 12 columnas"— y return df entrega la tabla para usarla después. Al final, df = cargar_datos("...") llama a la función y guarda el resultado en df.

In [ ]:
from pathlib import Path
import pandas as pd

# El notebook está en F2 y el dataset está en F1/data/raw
ARCHIVO = Path("../data/raw/ens2016.xlsx")

print("Directorio actual:")
print(Path.cwd())

print("\nRuta del archivo:")
print(ARCHIVO.resolve())

print("\n¿Existe el archivo?")
print(ARCHIVO.exists())

# Validación
if not ARCHIVO.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo en:\n{ARCHIVO.resolve()}"
    )

# Cargar dataset
df_crudo = pd.read_excel(
    ARCHIVO,
    na_values=["N/A"]
)

# Mantener copia original
df = df_crudo.copy()

print("\nDatos cargados correctamente")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

df.head()

Mostramos las primeras filas para confirmar visualmente que las columnas se cargaron correctamente:

`head()` muestra las primeras cinco filas. Es la comprobación más barata que existe: confirma que el separador se interpretó bien, que los nombres de columna son los esperados y que los valores no quedaron corridos de columna.

> **Qué mirar aquí.** Que `bmi` aparezca como número y no como texto. Si el archivo trae los faltantes escritos como `N/A`, pandas los reconoce; si vinieran como `--` o `sin dato`, la columna entera se cargaría como texto y todas las operaciones numéricas fallarían más adelante.

In [ ]:
df.head()

## 2. Exploración inicial

**Qué hace este paso.** Antes de modificar nada, miramos el estado original de los datos.
Esto nos da la "línea base" contra la cual verificaremos después cada transformación.

La función `explorar_dataframe` reporta cuatro cosas:
1. **Dimensiones** (`shape`): cuántas filas y columnas hay.
2. **Tipos de datos** (`dtypes`): qué columnas son numéricas y cuáles son texto. Las de
   texto (`object`) son las que tendremos que codificar más adelante.
3. **Valores nulos** (`isnull().sum()`): cuántos datos faltan en cada columna. Aquí
   detectaremos que `bmi` tiene huecos.
4. **Estadísticos descriptivos** (`describe()`): media, mínimo, máximo, etc., de las
   variables numéricas, útil para detectar rangos raros o valores atípicos.

In [ ]:
def explorar_dataframe(df):
    """Resumen exploratorio: dimensiones, tipos, nulos y estadisticos descriptivos."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos (variables numericas):")
    return df.describe()


explorar_dataframe(df)

También revisamos **cuántas categorías distintas** tiene cada variable de texto y cuántas
veces aparece cada una. Esto cumple dos funciones: detectar errores de tipeo o categorías
inesperadas, y conocer el orden en que aparecerán las categorías al codificarlas.

In [ ]:
# Conteo de categorias en las variables nominales (control de calidad de entrada)
columnas_categoricas = ['Zona']
for col in columnas_categoricas:
    print(f"\n{col}:")
    # dropna=False incluye los nulos como una categoría más: si una variable
    # tiene faltantes queremos verlos aquí, no que desaparezcan del conteo.
    print(df[col].value_counts(dropna=False))

### El diccionario de variables

Antes de limpiar hay que declarar **qué es cada variable**. Esta es la decisión
que determina todo el preprocesamiento posterior.

La idea central: **el tipo de dato no dice qué es una variable**. En este
conjunto `hypertension`, `id` y `age` son todas numéricas, y sin embargo exigen
tratamientos completamente distintos.

| Rol analítico | Qué preprocesamiento exige |
| --- | --- |
| **Continua** | Detección de atípicos, imputación, escalamiento |
| **Binaria** | Codificación 0/1 y revisión de desbalance |
| **Nominal** | *One-hot encoding* |
| **Ordinal** | Codificación con el orden declarado explícitamente |
| **Identificador** | Verificar unicidad y excluir del análisis |
| **Objetivo** | Se preserva sin transformar en esta fase |

In [ ]:
# El diccionario se DECLARA con lo que se sabe del dominio y se COMPLETA con lo
# observado en el archivo: no se afirma nada que no se verifique.
DECLARADO = [
    ("id",                "identificador", "Identificador único del paciente"),
    ("gender",            "nominal",       "Female, Male, Other"),
    ("age",               "continua",      "Edad en años"),
    ("hypertension",      "binaria",       "1 si presenta hipertensión diagnosticada"),
    ("heart_disease",     "binaria",       "1 si presenta cardiopatía"),
    ("ever_married",      "nominal",       "Yes / No"),
    ("work_type",         "nominal",       "Tipo de ocupación"),
    ("Residence_type",    "nominal",       "Urban / Rural"),
    ("avg_glucose_level", "continua",      "Glucosa promedio en sangre"),
    ("bmi",               "continua",      "Índice de masa corporal"),
    ("smoking_status",    "ordinal",       "Unknown < never < formerly < smokes"),
    ("stroke",            "objetivo",      "1 si el paciente presentó el evento"),
]
diccionario = pd.DataFrame(DECLARADO, columns=["variable", "rol", "descripcion"])

# Columnas OBSERVADAS: se leen del archivo, no se escriben a mano
diccionario["dtype"] = [str(df_crudo[v].dtype) for v in diccionario["variable"]]
# nunique() cuenta valores distintos: distingue una binaria de una continua
diccionario["n_unicos"] = [int(df_crudo[v].nunique()) for v in diccionario["variable"]]
# Sobre booleanos, mean() devuelve la PROPORCIÓN de True: por 100 da el porcentaje
diccionario["pct_nulos"] = [round(df_crudo[v].isna().mean() * 100, 1)
                            for v in diccionario["variable"]]

# El diccionario es un entregable, no una tabla de paso: se guarda
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
diccionario

### Valores atípicos: medir antes de decidir

La sección siguiente imputa con la mediana «porque hay valores extremos». Esa
afirmación hay que **demostrarla**, no enunciarla. Se usa el criterio del rango
intercuartílico:

$$\text{atípico si} \quad x < Q_1 - 1{,}5 \cdot \text{RIC} \quad \text{o} \quad x > Q_3 + 1{,}5 \cdot \text{RIC}$$

In [ ]:
def detectar_atipicos_iqr(serie, factor=1.5):
    """Identifica valores atípicos por el criterio del rango intercuartílico.

    Retorna
    -------
    tuple(np.ndarray, float, float)
        Máscara booleana de atípicos, límite inferior y límite superior.
    """
    # dropna() es obligatorio: un NaN propagaría y devolvería NaN como cuartil
    valores = serie.dropna().to_numpy(dtype="float64")

    # np.percentile con una lista devuelve varios percentiles de una vez
    q1, q3 = np.percentile(valores, [25, 75])
    ric = q3 - q1                                  # rango intercuartílico
    limite_inf, limite_sup = q1 - factor * ric, q3 + factor * ric

    # El operador | es el "o" elemento a elemento de pandas. Con 'or' fallaría:
    # 'or' espera un único booleano, no una serie completa.
    mascara = (serie < limite_inf) | (serie > limite_sup)
    # Comparar con NaN da False pero deja NA: fillna(False) lo hace explícito
    return mascara.fillna(False).to_numpy(), limite_inf, limite_sup


filas = []
for col in ["age", "avg_glucose_level", "bmi"]:
    mascara, inf, sup = detectar_atipicos_iqr(df[col])
    filas.append({
        "variable": col,
        "limite_inf": round(inf, 2),
        "limite_sup": round(sup, 2),
        "n_atipicos": int(mascara.sum()),          # sobre booleanos, sum() cuenta True
        "pct_atipicos": round(mascara.mean() * 100, 2),
        "media": round(df[col].mean(), 2),
        "mediana": round(df[col].median(), 2),
    })
pd.DataFrame(filas)

> **Cómo se lee esta tabla.** Donde la **media se aleja de la mediana** hay
> asimetría: la media queda arrastrada por los valores extremos. Ese es el
> argumento técnico —no una preferencia— para imputar con la mediana. Escríbanlo
> así en el informe, con las cifras de su propio conjunto.

## 3. Limpieza de datos

Limpiar significa dejar los datos completos y consistentes. Tomamos dos decisiones, cada
una justificada técnicamente:

**a) Eliminar `id`.** Es un identificador único (un número distinto por paciente). No
contiene información que ayude a predecir el accidente cerebrovascular, así que lo
descartamos para que no introduzca ruido.

**b) Imputar `bmi` con la mediana.** `bmi` (índice de masa corporal) es la única columna
con valores faltantes. "Imputar" es rellenar esos huecos con un valor representativo.
Tenemos dos opciones:
- La **media** (promedio): se ve arrastrada por los valores extremos.
- La **mediana** (valor central): es **robusta** ante valores atípicos.

Como el `bmi` tiene valores atípicos (se ven en el *boxplot* del final), usar la mediana
evita que esos extremos sesguen el relleno. La función `imputar_nulos_numericos` permite
elegir la estrategia con un parámetro e informa cuántos nulos rellenó y con qué valor
(trazabilidad).

In [ ]:
def imputar_nulos_numericos(df, columna, estrategia="mediana"):
    """
    Imputa los valores nulos de una columna numerica.

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna numerica a imputar.
    estrategia : str
        'media' o 'mediana'.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna imputada.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")
    # isnull() da True/False por celda; sum() cuenta los True.
    # Se calcula ANTES de imputar: después ya no habría nulos que contar.
    n_nulos = int(df[columna].isnull().sum())
    if estrategia == "media":
        valor = df[columna].mean()
    elif estrategia == "mediana":
        valor = df[columna].median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'")
    df = df.copy()
    df[columna] = df[columna].fillna(valor)
    print(f"'{columna}': {n_nulos} nulos imputados con la {estrategia} = {valor:.2f}")
    return df

**Antes** de limpiar, confirmamos dónde están los nulos:

In [ ]:
df.isnull().sum()

Eliminamos el identificador `id`:

In [ ]:
# Antes de descartarlo se comprueba que 'id' sea efectivamente un identificador:
# is_unique devuelve True si no hay valores repetidos.
assert df['id'].is_unique, "'id' tiene valores repetidos: revísalo antes de descartarlo"

# drop(columns=[...]) devuelve un DataFrame NUEVO sin esa columna.
df = df.drop(columns=['id'])
df.head()

Imputamos los nulos de `bmi` con la mediana:

In [ ]:
df = imputar_nulos_numericos(df, "bmi", estrategia="mediana")

### Comparar los métodos antes de elegir

La celda anterior aplica la mediana. Falta lo que la rúbrica evalúa: **por qué
esa y no otra**. Ninguna estrategia es mejor en abstracto.

| Método | Cuándo conviene | Qué distorsiona |
| --- | --- | --- |
| **Eliminar filas** | Faltantes escasos y aparentemente aleatorios | Pierde muestra; sesga si no son aleatorios |
| **Media** | Distribución aproximadamente simétrica | La arrastran los extremos; reduce la dispersión |
| **Mediana** | Hay valores extremos o asimetría | Reduce la dispersión; ignora las demás variables |
| **Mediana por grupo** | El valor depende de otra variable observada | Reduce la dispersión dentro de cada grupo |

In [ ]:
def comparar_imputaciones(datos, columna, agrupador=None, n_grupos=4):
    """Compara el efecto de cuatro estrategias de imputación sobre una variable.

    Retorna
    -------
    pd.DataFrame con el efecto de cada estrategia sobre la distribución.
    """
    original = datos[columna]
    if original.isna().sum() == 0:            # sin faltantes no hay nada que comparar
        raise ValueError(f"'{columna}' no tiene valores faltantes.")

    resultados = {
        "sin imputar (referencia)": original.dropna(),
        "eliminar filas": original.dropna(),
        "media": original.fillna(original.mean()),
        "mediana": original.fillna(original.median()),
    }
    if agrupador is not None:
        # qcut corta en tramos con la MISMA cantidad de casos (cuartiles si q=4);
        # duplicates="drop" evita el error cuando dos cortes coinciden.
        tramos = pd.qcut(datos[agrupador], q=n_grupos, duplicates="drop")
        # transform("median") devuelve una serie del MISMO largo, con la mediana
        # del grupo en cada fila. Con agg() se obtendría una fila por grupo,
        # que no sirve para rellenar.
        resultados["mediana por grupo"] = original.fillna(
            original.groupby(tramos, observed=True).transform("median"))

    filas = [{
        "estrategia": nombre,
        "n": len(serie.dropna()),
        "media": round(serie.mean(), 3),
        "desv_est": round(serie.std(), 3),
        "asimetria": round(serie.skew(), 3),     # skew() mide la cola de la distribución
    } for nombre, serie in resultados.items()]

    tabla = pd.DataFrame(filas)
    referencia = tabla.iloc[0]["desv_est"]       # iloc[0] es la fila de referencia
    # La columna decisiva: cuánto deforma cada método la dispersión original
    tabla["cambio_desv_%"] = ((tabla["desv_est"] - referencia) / referencia * 100).round(2)
    tabla["filas_perdidas"] = len(datos) - tabla["n"]
    return tabla


comparar_imputaciones(df_crudo, "bmi", agrupador="age")

**Cómo se lee.** `cambio_desv_%` es la columna que decide: toda imputación por
un valor central concentra los datos y **reduce artificialmente la desviación
estándar**. El método que menos la reduce es el que menos deforma la variable.
`filas_perdidas` muestra el costo de eliminar, y `asimetria` indica si la media
representa bien el centro.

> **Si las filas dan casi lo mismo**, la variable es simétrica y las estrategias
> son equivalentes: lo correcto es declararlo y elegir la más simple. Escribir
> «se descarta la media por la asimetría» cuando la tabla muestra asimetría nula
> es exactamente la incoherencia que la rúbrica penaliza.

In [ ]:
# La bandera de imputación: que un valor faltara puede ser informativo en sí
# mismo, de modo que el hecho se conserva en una columna aparte.
df["bmi_imputado"] = df_crudo["bmi"].isna().astype(int)   # 1 = valor imputado
print(f"Filas marcadas como imputadas: {int(df['bmi_imputado'].sum())}")
print(f"Desviación estándar de 'bmi': "
      f"{df_crudo['bmi'].std():.3f} -> {df['bmi'].std():.3f}")

> **Atención — fuga de datos.** Aquí el valor de relleno se calcula sobre todo
> el conjunto porque no hay partición entre entrenamiento y prueba. Cuando la
> haya, debe calcularse **solo con el conjunto de entrenamiento**: usar la
> mediana global filtra información del conjunto de prueba e infla los
> resultados. El mismo cuidado aplica al escalamiento.

**Después** de limpiar, verificamos que ya no quede ningún nulo (comprobación intermedia):

In [ ]:
df.isnull().sum()

## 4. Transformación: codificación One-Hot

Los modelos solo entienden números, no texto. Por eso hay que convertir las variables de
texto en números. Pero **cómo** las convertimos importa:

- `gender`, `ever_married`, `work_type`, `Residence_type` y `smoking_status` son
  **categóricas nominales**: sus categorías **no tienen un orden** ("Urbano" no es mayor ni
  menor que "Rural").
- Si simplemente les pusiéramos 0, 1, 2..., el modelo creería que existe un orden falso
  (que "2" vale más que "1"). Para evitarlo usamos **One-Hot Encoding**: cada categoría se
  convierte en su propia columna de 0 y 1.

Por ejemplo, la columna `Residence_type` se convierte en dos columnas:

| Residence_type | → | res_Rural | res_Urban |
|---|---|---|---|
| Urban | | 0 | 1 |
| Rural | | 1 | 0 |

Mantenemos el método visto en el curso —**`LabelEncoder` seguido de `OneHotEncoder`**— pero
lo encapsulamos en una sola función para no repetir el mismo bloque cinco veces.

### El patrón `fit` / `transform`

Casi todas las herramientas de scikit-learn funcionan en dos tiempos:
- **`fit`** = *aprender*: la herramienta mira los datos y guarda lo que necesita (por
  ejemplo, qué categorías existen).
- **`transform`** = *aplicar*: usa lo aprendido para convertir los datos.

Dentro de la función `codificar_one_hot` este patrón aparece **dos veces** (una con
`LabelEncoder` y otra con `OneHotEncoder`). Paso a paso, con el ejemplo `["Yes","No","Yes","No"]`:

1. **`le = preprocessing.LabelEncoder()`** — crea la herramienta que convierte texto en
   enteros. Aún está vacía.
2. **`le.fit(datos)`** — aprende las categorías y les asigna un número en orden alfabético:
   `"No" → 0`, `"Yes" → 1`.
3. **`le.transform(datos)`** — aplica lo aprendido → `[1, 0, 1, 0]`. Esto ya es *Label
   Encoding*.
4. **`ohe = preprocessing.OneHotEncoder()`** — crea la segunda herramienta, para One-Hot.
5. **`d = datos_codificados.reshape(-1, 1)`** — reorganiza los datos de una fila a una
   columna, porque el `OneHotEncoder` exige formato de tabla (2 dimensiones). El `-1`
   significa "calcula tú las filas" y el `1` significa "una columna".
6. **`ohe.fit(d)` y `ohe.transform(d).toarray()`** — expande cada entero en columnas
   binarias. `.toarray()` convierte el resultado a un arreglo normal.
7. **`pd.DataFrame(...)` + `pd.concat(...)`** — empaqueta las nuevas columnas con nombres
   legibles y las une al `DataFrame`, eliminando la columna original.

**Mejoras respecto al código original:** la función (a) asigna los nombres legibles
directamente, en lugar de renombrar por número de posición; (b) conserva el índice original
(`index=df.index`) para que la unión no desalinee filas; y (c) **valida** que la cantidad
de nombres coincida con la cantidad de categorías, avisando con un error claro si no.

In [ ]:
def codificar_one_hot(df, columna, nombres_columnas):
    """
    Codifica una variable categorica nominal aplicando LabelEncoder y luego
    OneHotEncoder (patron fit/transform de scikit-learn).

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna categorica a codificar.
    nombres_columnas : list[str]
        Nombres de las nuevas columnas binarias. El orden debe coincidir con el
        orden alfabetico de las categorias (criterio interno de LabelEncoder).

    Retorna
    -------
    pd.DataFrame
        DataFrame con las nuevas columnas one-hot y sin la columna original.

    Lanza
    -----
    KeyError
        Si la columna no existe.
    ValueError
        Si el numero de nombres no coincide con el numero de categorias.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")

    # Paso 1-3: LabelEncoder aprende las categorias y las convierte a enteros
    le = preprocessing.LabelEncoder()
    datos = df[columna]
    le.fit(datos)
    datos_codificados = le.transform(datos)

    # Verificacion: los nombres deben corresponder a las categorias detectadas
    if len(nombres_columnas) != len(le.classes_):
        raise ValueError(
            f"Se esperaban {len(le.classes_)} nombres para '{columna}' "
            f"({list(le.classes_)}), pero se recibieron {len(nombres_columnas)}."
        )

    # Paso 4-6: OneHotEncoder expande los enteros a columnas binarias
    ohe = preprocessing.OneHotEncoder()
    d = datos_codificados.reshape(-1, 1)   # formato 2D requerido por el encoder
    ohe.fit(d)
    matriz = ohe.transform(d).toarray()

    # Paso 7: empaquetamos con nombres legibles y conservamos el indice original
    nuevas = pd.DataFrame(matriz, columns=nombres_columnas, index=df.index).astype(int)

    df = df.drop(columns=[columna]).reset_index(drop=True)
    nuevas = nuevas.reset_index(drop=True)
    return pd.concat([df, nuevas], axis=1)